# Lab 5, Day 1 — Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [ ]:
import pandas as pd
import numpy as np

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


## Step 1: Load and profile

In [ ]:
# TODO: df.info(), df.describe(include='all').T, and missingness percentage per
# column, sorted descending
df.info()

df.describe(include="all").T

missing = (df.isna().mean() * 100).round(2)
print("Missing %:")
print(missing.sort_values(ascending=False))

print("\nSurvival counts:")
print(df["Survived"].value_counts())

print("\nSurvival proportion:")
print(df["Survived"].value_counts(normalize=True).round(3))



## Step 2: The most skipped checks - does missingness predict the target?

In [ ]:
# TODO: for at least Age and Cabin, compare Survived rates between rows where the
# column is missing vs not. Is there a difference worth caring about?
print("Age missing vs present:")
print(df.groupby(df["Age"].isna()).agg(rows=("Survived", "size"),survival_rate=("Survived", "mean")).round(3))

print("\nCabin missing vs present:")
print(df.groupby(df["Cabin"].isna()).agg(rows=("Survived", "size"),survival_rate=("Survived", "mean")).round(3))


## Step 3: Look for impossible values and placeholders

In [ ]:
# TODO: check unique values in object columns, zero/negative fares, absurd ages -
# anything that looks like a placeholder rather than real missingness
print("Object columns:")
for col in df.select_dtypes(include="object").columns:
    print(col, df[col].unique()[:20])

print("\nFare checks:")
print("Zero:", (df["Fare"] == 0).sum())
print("Negative:", (df["Fare"] < 0).sum())

print("\nAge checks:")
print("Negative:", (df["Age"] < 0).sum())
print("Over 100:", (df["Age"] > 100).sum())
print("Min age:", df["Age"].min())
print("Max age:", df["Age"].max())

placeholders = ["?", "unknown", "Unknown", "NA", "N/A", "none", "None", ""]

print("\nPlaceholder check:")
for col in df.select_dtypes(include="object").columns:
    n = df[col].isin(placeholders).sum()
    if n > 0:
        print(col, n)


## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

**Cleaning decisions:**

For `Age`, I am keeping the column and will fill the missing values later. The missingness check also shows that whether age is recorded can carry some information, so I will test an `age_missing` feature.

For `Cabin`, most values are missing and the full cabin number has many categories. I will test a simpler `has_cabin` feature instead of using the raw cabin value in the engineered model.

For `Embarked`, only a very small number of values are missing, so I will fill them using the most common category.

For `Fare`, I checked for missing, zero, and negative values. I will keep the valid values as they are and let the preprocessing pipeline impute any actual missing fare values.

For `Name`, I do not want to use the full name directly in the engineered model because almost every value is unique. I will keep it long enough to extract the passenger title.

For `Ticket` and `home.dest`, I am dropping the raw columns because they have many unique values and are not features I want to encode directly.

`PassengerId`, when it exists in the fallback dataset, is also dropped because it is just an identifier.

For `boat` and `body`, I will drop both because they contain information that would only be known after the outcome and would cause target leakage.

I also checked the object columns for obvious placeholder values such as `?`, `unknown`, `N/A`, or empty strings.


## Step 5: Split first, before fitting anything (`pipeline.py`)

In [ ]:
from sklearn.model_selection import train_test_split

# PassengerId only exists in the synthetic fallback, so errors="ignore"
# keeps the same code working with either data source.
X = df.drop(
    columns=["Survived", "PassengerId", "boat", "body", "Ticket", "home.dest"],
    errors="ignore"
)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print("train survival rate:", round(y_train.mean(), 3))
print("test survival rate:", round(y_test.mean(), 3))


## Step 6: Column groups - and check them by eye

In [ ]:
# TODO: split X's columns into numeric vs categorical. Print both lists and look at
# them - does anything dtype-based selection picked up actually belong in the other
# group? (Think about what Pclass really represents.)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Pclass looks numeric in pandas, but it is really a category
num_cols.remove("Pclass")
cat_cols.append("Pclass")

print("numeric:", num_cols)
print("categorical:", cat_cols)


In [ ]:
import joblib

split = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test
}

joblib.dump(split, "split.joblib")
print("saved split.joblib")


---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.